# 🦜 LangChain — Build RAG Faster

## What is LangChain?

LangChain is a framework that gives you pre-built components for RAG:
- Document loaders (PDF, web, CSV...)
- Text splitters (chunk documents automatically)
- Vector stores (Chroma, FAISS, Pinecone...)
- LLM wrappers (OpenAI, Anthropic, local...)
- Chains (connect all the pieces)

## From Scratch vs LangChain

```python
# From scratch (what we've been doing):
embeddings = model.encode(docs)
scores     = cosine_similarity([q_emb], embeddings)[0]
# ... 20+ lines to wire everything together

# With LangChain:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())
answer = chain.invoke("Your question")   ← done!
```

LangChain shines in production. For learning, doing it from scratch first (like this course) helps you understand what's happening.

In [1]:
# !pip install langchain langchain-community chromadb sentence-transformers

In [2]:
# Step 1: Load documents
from langchain.schema import Document

# Normally you'd use a loader:
# from langchain_community.document_loaders import TextLoader
# docs = TextLoader("my_file.txt").load()

# Here: create documents manually
raw_docs = [
    Document(page_content="RAG combines retrieval with LLM generation to answer questions from your data."),
    Document(page_content="Embeddings are vectors that represent the meaning of text in a high-dimensional space."),
    Document(page_content="Chunking splits long documents into smaller pieces so they fit in an LLM context window."),
    Document(page_content="Vector databases store embeddings and support fast similarity search."),
    Document(page_content="Reranking re-scores retrieved documents for higher precision before passing to the LLM."),
]

print(f"Loaded {len(raw_docs)} documents")

Loaded 5 documents


In [3]:
# Step 2: Split into chunks (for longer docs)
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # max characters per chunk
    chunk_overlap=20,     # overlap to preserve context at boundaries
)

chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

Split into 5 chunks


In [4]:
# Step 3: Create a vector store
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# This embeds all chunks and stores them in Chroma (local vector DB)
vectorstore = Chroma.from_documents(chunks, embedding_model)

print(f"Vector store created with {vectorstore._collection.count()} documents")

/var/folders/d8/1ndp4tcd1mqf1hvm9yld50z00000gn/T/ipykernel_21367/1614138610.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Vector store created with 5 documents


In [5]:
# Step 4: Search with a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

question = "What are embeddings?"
results  = retriever.invoke(question)

print(f"Query: '{question}'\n")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

Query: 'What are embeddings?'

  1. Embeddings are vectors that represent the meaning of text in a high-dimensional space.
  2. Vector databases store embeddings and support fast similarity search.


In [6]:
# Step 5: Full RAG chain with an LLM
# Uncomment to use with a real LLM

langchain_rag_code = """
from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA

llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

result = chain.invoke({"query": "What is chunking?"})
print(result["result"])          # The answer
print(result["source_documents"]) # What it used
"""

print("LangChain RAG with LLM:")
print(langchain_rag_code)

LangChain RAG with LLM:

from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA

llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

result = chain.invoke({"query": "What is chunking?"})
print(result["result"])          # The answer
print(result["source_documents"]) # What it used



## LangChain at a Glance

```
Load docs     → TextLoader, PyPDFLoader, WebBaseLoader...
Split chunks  → RecursiveCharacterTextSplitter
Embed + Store → Chroma, FAISS, Pinecone, Weaviate...
Retrieve      → vectorstore.as_retriever()
Generate      → RetrievalQA, ConversationalRetrievalChain
```

**Docs:** https://python.langchain.com  
**Next:** `01_LlamaIndex_RAG.ipynb` — LangChain's main competitor